# AI DataOps Guardian — Notebook docente

**Objetivo:** observar diferencias entre script, pipeline y agente usando datos sintéticos. Ejecutar desde la carpeta raíz con `jupyter lab` (si está instalado) o copiar celdas a Colab con los archivos del proyecto.


## 1. Importaciones y primer dataset
El dataset no contiene información privada: es sintético y reproducible.


In [ ]:
from backend.app.data import sample, FEATURES, profile

df = sample(10, seed=42)
print("Columnas:", list(df.columns))
print("Filas:", len(df))
print(df.head())


## 2. Script rígido
Cambiamos una columna para provocar un error. El script no tiene políticas de recuperación.


In [ ]:
from backend.app.engine import script_run
r = script_run("schema_change")
print("Estado:", r.status)
print("Error:", r.summary)


## 3. Pipeline de calidad
No improvisamos: el pipeline detiene inferencia si los datos no cumplen el esquema.


In [ ]:
from backend.app.engine import pipeline_run
r = pipeline_run("null_values")
for step in r.events:
    print(step["name"], "→", step["detail"])


## 4. Drift
El PSI compara distribuciones; por sí solo no mide caída de accuracy.


In [ ]:
r = pipeline_run("drift")
print("Estado:", r.status)
print("PSI:", r.metrics.get("max_psi"))


## 5. Human-in-the-loop
Una propuesta nunca equivale a autorización: se requiere aprobación explícita.


In [ ]:
from backend.app.schemas import RunRequest
from backend.app.service import execute

sin_aprobar = execute(RunRequest(scenario="schema_change", mode="agent", use_llm=False))
print("Sin aprobar:", sin_aprobar["status"], sin_aprobar["proposal"])

con_aprobacion = execute(RunRequest(scenario="schema_change", mode="agent", use_llm=False, approve_action=True))
print("Con aprobación:", con_aprobacion["status"])


## 6. Debate de cierre
- ¿Qué resolvió el pipeline sin usar ningún LLM?
- ¿En qué situación sí ayuda el razonamiento de un agente?
- ¿Por qué el agente no puede ejecutar cualquier comando?
- ¿Qué evidencia pide el checkpoint de Docker?
